In [ ]:
### Imports

import torch
import rasterio
import numpy as np
import torch.nn as nn
import glob
import re
import pandas as pd
import torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import os
from datetime import datetime

In [ ]:
# Define the 9 reservoirs
dam_names = ['baells', 'boadella', 'foix', 'llosa de cavall', 'riudecanyes', 'sant ponç', 'sau', 'siurana', 'susqueda']

# Define the 2 loss models
losses = ['log_mse', 'mape']

# Define the 2 out-of-distribution dams
dam_test = ['escales', 'terradets']

# Device
device = torch.device('cpu')
if torch.cuda.is_available(): device = torch.device('cuda')
if torch.backends.mps.is_available(): device = torch.device("mps")
print (f"Using device: {device}")

# Define the architecture
class DamCNN(nn.Module):
    def __init__(self):
        super(DamCNN, self).__init__()

        # Block 1: Input is (1, 500, 500) -> Output is (16, 250, 250)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Block 2: Input is (16, 250, 250) -> Output is (32, 125, 125)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Block 3: Input is (32, 125, 125) -> Output is (64, 62, 62)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Block 4: Input is (64, 62, 62) -> Output is (128, 31, 31)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2)

        # Destroy spatial memorization.
        # It takes 128 maps of 31x31 pixels and turns them into exactly 128 numbers.
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        
        self.fc1 = nn.Linear(128, 256)
        self.relu_fc = nn.ReLU()
        
        # Final layer outputs a single number (the volume)
        self.fc2 = nn.Linear(in_features=256, out_features=1)
        
    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.pool4(self.relu4(self.conv4(x)))
        
        x = self.global_pool(x)
        x = self.flatten(x)
        
        x = self.relu_fc(self.fc1(x))
        x = self.fc2(x)
        
        return x.view(-1, 1)

# Ensemble predictive function
def predict_ensemble(image_path, models_list, device, loss, max_size=500):
    with rasterio.open(image_path) as src:
        img = src.read(1)
        
    # Clean the data
    img = np.nan_to_num(img, nan=-1.0)
    img = np.clip(img, -1.0, 1.0)
    
    # Convert to Tensor and pad
    tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0).float()
    pad_h = max_size - tensor.shape[2]
    pad_w = max_size - tensor.shape[3]
    padded_tensor = F.pad(tensor, (0, pad_w, 0, pad_h), value=-1.0).to(device)

    raw_volumes = []
    
    # Pass through all models
    with torch.no_grad():
        for model in models_list:
            # The model outputs a fraction (e.g., 0.15)
            pred_scaled = model(padded_tensor).item() 
            
            if loss == 'log_mse':
                # Reverse log(V+1) -> exp(x) - 1
                pred_raw = np.expm1(pred_scaled)
            else:
                # Reverse the linear 300.0 scaling
                pred_raw = pred_scaled * 300.0
            
            raw_volumes.append(pred_raw)
            
    # Calculate statistics
    final_prediction = np.mean(raw_volumes)
    uncertainty = np.std(raw_volumes) 
    
    return final_prediction, uncertainty, raw_volumes

# Savgol Filter
def predict_and_filter_timeseries(chronological_image_paths, models_list, device, loss):

    timeline_predictions = []
    timeline_uncertainties = []
    
    print(f"Processing {len(chronological_image_paths)} images over time...")
    
    for img_path in chronological_image_paths:
        # Get the ensemble prediction for this specific date
        mean_vol, std_vol, _ = predict_ensemble(img_path, models_list, device, loss)
        
        timeline_predictions.append(mean_vol)
        timeline_uncertainties.append(std_vol)
        
    # Convert to numpy arrays for filtering
    raw_curve = np.array(timeline_predictions)
    
    # Apply the Savgol filter
    smoothed_curve = savgol_filter(raw_curve, window_length=7, polyorder=2)
    
    # Prevent the filter from mathematically dipping below 0 hm3
    smoothed_curve = np.clip(smoothed_curve, a_min=0.0, a_max=None)
    
    return raw_curve, smoothed_curve, timeline_uncertainties

Using device: cpu


In [ ]:
# Loop over the the test dams and the specified losses
# (log_mse and mape), and call the filtered prediction
# function  
for test in dam_test:
    print('='*20)
    print(f'Testing on {test}')
    print('='*20)
    for loss in losses:
        ensemble = []
        for dam in dam_names:
            model = DamCNN().to(device)
            
            model_path = f'../models/dam_model_loocv_{dam}_{loss}.pth' 
            
            if not os.path.exists(model_path):
                print(f"⚠️ ERROR: Could not find {model_path}. Check your folder structure.")
                continue
                
            model.load_state_dict(torch.load(model_path, map_location=device))
            model.eval() 
            
            ensemble.append(model)
        
        # Grab all the TIFFs for a specific dam and sort them chronologically
        image_folder = f"../data/{test}/*.tiff" 
        chronological_paths = sorted(glob.glob(image_folder)) 

        if len(chronological_paths) > 0:
            raw_vols, smooth_vols, uncertainties = predict_and_filter_timeseries(chronological_paths, ensemble, device, loss)
        
        # Load the CSV
        labels_df = pd.read_csv(f"../data/{test}/volume_{test}/volume_{test}.csv", sep=';')

        # Convert date to datetime
        labels_df['FECHA_GRUPO'] = pd.to_datetime(labels_df['FECHA_GRUPO'], format='%d/%m/%Y')

        true_volumes_ordered = []

        # Extract the data from the file name
        def extract_date_from_filename(filename):
            # This regex looks for YYYY-MM-DD or YYYY_MM_DD anywhere in the filename
            match = re.search(r'(\d{4})[-_](\d{2})[-_](\d{2})', filename)
            if match:
                year, month, day = match.groups()
                return datetime(int(year), int(month), int(day))
            return None

        found_dates = []

        for img_path in chronological_paths:
            filename = os.path.basename(img_path)
            img_date = extract_date_from_filename(filename)
            
            if img_date is not None:
                # Search the dataframe for this specific date
                match = labels_df[labels_df['FECHA_GRUPO'] == img_date]
                
                if not match.empty:
                    # Grab the 'ACUMULADO (hm3)' column
                    true_volume = match['ACUMULADO (hm3)'].values[0]
                    true_volumes_ordered.append(true_volume)
                    found_dates.append(img_date)
                else:
                    print(f"⚠️ No CSV label found for image date: {img_date}")
                    true_volumes_ordered.append(np.nan) 
            else:
                print(f"⚠️ Could not extract a date from filename: {filename}")
                true_volumes_ordered.append(np.nan)

        # Create the final DataFrame
        df_results = pd.DataFrame({
            "filename": [os.path.basename(p) for p in chronological_paths],
            "date": found_dates,
            "true_volume": true_volumes_ordered,
            "raw_pred": raw_vols,
            "smooth_pred": smooth_vols,
            "uncertainties": uncertainties
        })

        df_results.to_csv(f'../output/tables/ensemble_{loss}_predictions_{test}.csv', index=False)
        print(f"✅ {test}, {loss} -- Results and true labels matched and saved successfully!")

        valid_df = df_results.dropna(subset=['true_volume'])

        if len(valid_df) > 0:
            # Calculate Absolute Errors in hm³
            raw_mae = np.mean(np.abs(valid_df['true_volume'] - valid_df['raw_pred']))
            smooth_mae = np.mean(np.abs(valid_df['true_volume'] - valid_df['smooth_pred']))
            
            # Calculate Percentage Errors in %
            # We divide the absolute error by the true volume, then multiply by 100
            raw_mape = np.mean(np.abs((valid_df['true_volume'] - valid_df['raw_pred']) / valid_df['true_volume'])) * 100
            smooth_mape = np.mean(np.abs((valid_df['true_volume'] - valid_df['smooth_pred']) / valid_df['true_volume'])) * 100
            
            # 3. Print the results
            print(f"\n--- {test.capitalize()} STRESS TEST RESULTS ---")
            print(f"Raw Predictions MAE:     {raw_mae:.2f} hm³  |  Error: {raw_mape:.2f}%")
            print(f"Smoothed (Savgol) MAE:   {smooth_mae:.2f} hm³  |  Error: {smooth_mape:.2f}%")
            print("\n")

Testing on escales
✅ All 9 models successfully loaded and ready for inference!
Processing 217 images over time...
✅ escales, log_mse -- Results and true labels matched and saved successfully!

--- Escales STRESS TEST RESULTS ---
Raw Predictions MAE:     57.05 hm³  |  Error: 53.38%
Smoothed (Savgol) MAE:   56.96 hm³  |  Error: 53.16%


✅ All 9 models successfully loaded and ready for inference!
Processing 217 images over time...
✅ escales, mape -- Results and true labels matched and saved successfully!

--- Escales STRESS TEST RESULTS ---
Raw Predictions MAE:     46.11 hm³  |  Error: 42.93%
Smoothed (Savgol) MAE:   45.98 hm³  |  Error: 42.62%


Testing on terradets
✅ All 9 models successfully loaded and ready for inference!
Processing 154 images over time...
✅ terradets, log_mse -- Results and true labels matched and saved successfully!

--- Terradets STRESS TEST RESULTS ---
Raw Predictions MAE:     15.59 hm³  |  Error: 49.30%
Smoothed (Savgol) MAE:   15.60 hm³  |  Error: 49.32%


✅ All